# Notebook 03 — Intégration des données

## Objectif

L'objectif de ce notebook est de combiner les différentes tables nettoyées
afin de créer des datasets analytiques adaptés à l'analyse des patients,
des exercices, des sessions et de leur progression.

Nous allons créer principalement :

1. exercise_analysis
2. patient_exercise_analysis
3. session_analysis

Ces datasets serviront ensuite à l'EDA, au calcul des KPI,
à l'analyse de progression et à la préparation du dashboard thérapeute.

In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

In [4]:
patients = pd.read_csv(PROCESSED_PATH + "patients.csv")
games = pd.read_csv(PROCESSED_PATH + "games.csv")
exercises = pd.read_csv(PROCESSED_PATH + "exercises.csv")
exercise_levels = pd.read_csv(PROCESSED_PATH + "exercise_levels.csv")
patient_exercises = pd.read_csv(PROCESSED_PATH + "patient_exercises.csv")
level_changes = pd.read_csv(PROCESSED_PATH + "level_changes.csv")
sessions = pd.read_csv(PROCESSED_PATH + "sessions.csv")
results = pd.read_csv(PROCESSED_PATH + "results.csv")

In [5]:
datasets = {
    "patients": patients,
    "games": games,
    "exercises": exercises,
    "exercise_levels": exercise_levels,
    "patient_exercises": patient_exercises,
    "level_changes": level_changes,
    "sessions": sessions,
    "results": results
}

for name, df in datasets.items():
    print(
        f"{name:20} → "
        f"{df.shape[0]:4} lignes | "
        f"{df.shape[1]:2} colonnes"
    )

patients             →   20 lignes |  7 colonnes
games                →    3 lignes |  4 colonnes
exercises            →    8 lignes |  4 colonnes
exercise_levels      →   24 lignes |  7 colonnes
patient_exercises    →   43 lignes |  8 colonnes
level_changes        →   73 lignes |  8 colonnes
sessions             →  500 lignes |  6 colonnes
results              →  500 lignes |  5 colonnes


In [6]:
exercise_analysis = exercises.merge(
    games[["id", "name", "body_part", "description"]],
    left_on="game_id",
    right_on="id",
    how="left",
    suffixes=("_exercise", "_game")
)

In [7]:
print("Avant le merge :", exercises.shape)
print("Après le merge :", exercise_analysis.shape)

Avant le merge : (8, 4)
Après le merge : (8, 8)


In [8]:
print(exercise_analysis.columns.tolist())

['id_exercise', 'game_id', 'name_exercise', 'description_exercise', 'id_game', 'name_game', 'body_part', 'description_game']


In [9]:
display(exercise_analysis.head())

,id_exercise,game_id,name_exercise,description_exercise,id_game,name_game,body_part,description_game
0,1,1,Forward Arm Raise,Raise the arm forward to shoulder height and a...,1,Bubble Reach,Shoulder,Arcade game where the child pops floating bubb...
1,2,1,Lateral Arm Raise,Raise the arm sideways to reach bubbles on the...,1,Bubble Reach,Shoulder,Arcade game where the child pops floating bubb...
2,3,1,Overhead Reach,Reach above the head to pop the highest bubbles,1,Bubble Reach,Shoulder,Arcade game where the child pops floating bubb...
3,4,2,Seated Knee Extension,Extend the knee from a seated position to kick...,2,Step Quest,Knee,Adventure game where the child controls a hero...
4,5,2,Mini Squat,Perform a shallow squat to duck under obstacles,2,Step Quest,Knee,Adventure game where the child controls a hero...


In [10]:
print(
    "Exercices sans jeu correspondant :",
    exercise_analysis["id_game"].isna().sum()
)

Exercices sans jeu correspondant : 0


In [13]:
print(
    "Nombre d'exercices uniques avant :",
    exercises["id"].nunique()
)

print(
    "Nombre d'exercices uniques après :",
    exercise_analysis["id_exercise"].nunique()
)

Nombre d'exercices uniques avant : 8
Nombre d'exercices uniques après : 8


In [14]:
patient_exercises_analysis = patient_exercises.copy()

In [15]:
patient_exercises_analysis = patient_exercises_analysis.merge(
    patients[
        ["id", "first_name", "last_name", "date_of_birth"]
    ],
    left_on="patient_id",
    right_on="id",
    how="left",
    suffixes=("", "_patient")
)

In [16]:
print("Nombre d'affectations avant :", patient_exercises.shape[0])
print("Nombre d'affectations après :", patient_exercises_analysis.shape[0])

Nombre d'affectations avant : 43
Nombre d'affectations après : 43


In [17]:
print(
    "Affectations sans patient correspondant :",
    patient_exercises_analysis["id_patient"].isna().sum()
)

Affectations sans patient correspondant : 0


In [18]:
print(patient_exercises_analysis.columns.tolist())

['id', 'patient_id', 'exercise_id', 'start_level_id', 'current_level_id', 'assigned_by', 'assigned_at', 'is_active', 'id_patient', 'first_name', 'last_name', 'date_of_birth']


In [19]:
patient_exercises_analysis = patient_exercises_analysis.merge(
    exercises[
        ["id", "game_id", "name", "description"]
    ],
    left_on="exercise_id",
    right_on="id",
    how="left",
    suffixes=("", "_exercise")
)

In [20]:
print(
    "Nombre d'affectations avant :",
    patient_exercises_analysis.shape[0]
)

Nombre d'affectations avant : 43


In [21]:
print(
    "Nombre de lignes après ajout des exercices :",
    patient_exercises_analysis.shape[0]
)

Nombre de lignes après ajout des exercices : 43


In [22]:
print(
    "Affectations sans exercice correspondant :",
    patient_exercises_analysis["id_exercise"].isna().sum()
)

Affectations sans exercice correspondant : 0


In [23]:
print(patient_exercises_analysis.columns.tolist())

['id', 'patient_id', 'exercise_id', 'start_level_id', 'current_level_id', 'assigned_by', 'assigned_at', 'is_active', 'id_patient', 'first_name', 'last_name', 'date_of_birth', 'id_exercise', 'game_id', 'name', 'description']


In [24]:
display(
    patient_exercises_analysis[
        [
            "patient_id",
            "first_name",
            "last_name",
            "exercise_id",
            "id_exercise",
            "game_id",
            "name"
        ]
    ].head()
)

,patient_id,first_name,last_name,exercise_id,id_exercise,game_id,name
0,1,Yasmine,Bouzid,7,7,3,Wrist Flexion and Extension
1,1,Yasmine,Bouzid,8,8,3,Forearm Rotation
2,3,Fatma,Tlili,8,8,3,Forearm Rotation
3,20,Omar,Dridi,1,1,1,Forward Arm Raise
4,11,Sirine,Ben Salah,5,5,2,Mini Squat


In [25]:
patient_exercises_analysis = patient_exercises_analysis.merge(
    games[
        ["id", "name", "body_part", "description"]
    ],
    left_on="game_id",
    right_on="id",
    how="left",
    suffixes=("", "_game")
)

In [26]:
print(
    "Nombre de lignes après ajout des jeux :",
    patient_exercises_analysis.shape[0]
)

Nombre de lignes après ajout des jeux : 43


In [27]:
print(
    "Affectations sans jeu correspondant :",
    patient_exercises_analysis["id_game"].isna().sum()
)

Affectations sans jeu correspondant : 0


In [29]:
print(patient_exercises_analysis.columns.tolist())

['id', 'patient_id', 'exercise_id', 'start_level_id', 'current_level_id', 'assigned_by', 'assigned_at', 'is_active', 'id_patient', 'first_name', 'last_name', 'date_of_birth', 'id_exercise', 'game_id', 'name', 'description', 'id_game', 'name_game', 'body_part', 'description_game']


In [28]:
display(
    patient_exercises_analysis[
        [
            "patient_id",
            "first_name",
            "last_name",
            "exercise_id",
            "name",
            "game_id",
            "name_game",
            "body_part"
        ]
    ].head(10)
)

,patient_id,first_name,last_name,exercise_id,name,game_id,name_game,body_part
0,1,Yasmine,Bouzid,7,Wrist Flexion and Extension,3,Wrist Wizard,Wrist
1,1,Yasmine,Bouzid,8,Forearm Rotation,3,Wrist Wizard,Wrist
2,3,Fatma,Tlili,8,Forearm Rotation,3,Wrist Wizard,Wrist
3,20,Omar,Dridi,1,Forward Arm Raise,1,Bubble Reach,Shoulder
4,11,Sirine,Ben Salah,5,Mini Squat,2,Step Quest,Knee
5,17,Skander,Karoui,3,Overhead Reach,1,Bubble Reach,Shoulder
6,9,Adam,Mejri,6,Step Up,2,Step Quest,Knee
7,19,Mohamed,Slimani,4,Seated Knee Extension,2,Step Quest,Knee
8,11,Sirine,Ben Salah,6,Step Up,2,Step Quest,Knee
9,20,Omar,Dridi,6,Step Up,2,Step Quest,Knee


In [30]:
patient_exercises_analysis = patient_exercises_analysis.merge(
    exercise_levels[
        [
            "id",
            "level_number",
            "title",
            "target_angle",
            "required_score",
            "required_progression"
        ]
    ],
    left_on="current_level_id",
    right_on="id",
    how="left",
    suffixes=("", "_current_level")
)

In [31]:
print(
    "Nombre de lignes après ajout du niveau :",
    patient_exercises_analysis.shape[0]
)

Nombre de lignes après ajout du niveau : 43


In [32]:
print(
    "Affectations sans niveau actuel correspondant :",
    patient_exercises_analysis["id_current_level"].isna().sum()
)

Affectations sans niveau actuel correspondant : 0


In [33]:
level_exercise_check = patient_exercises_analysis[
    patient_exercises_analysis["id_current_level"].notna()
][
    ["exercise_id", "id_current_level"]
].merge(
    exercise_levels[
        ["id", "exercise_id"]
    ],
    left_on="id_current_level",
    right_on="id",
    how="left",
    suffixes=("_assignment", "_level")
)

In [34]:
invalid_level_exercises = level_exercise_check[
    level_exercise_check["exercise_id_assignment"]
    !=
    level_exercise_check["exercise_id_level"]
]

print(
    "Niveaux appartenant à un mauvais exercice :",
    len(invalid_level_exercises)
)

Niveaux appartenant à un mauvais exercice : 0


In [35]:
display(
    patient_exercises_analysis[
        [
            "patient_id",
            "first_name",
            "last_name",
            "exercise_id",
            "name",
            "name_game",
            "current_level_id",
            "level_number",
            "title",
            "target_angle",
            "required_score",
            "required_progression"
        ]
    ].head(10)
)

,patient_id,first_name,last_name,exercise_id,name,name_game,current_level_id,level_number,title,target_angle,required_score,required_progression
0,1,Yasmine,Bouzid,7,Wrist Flexion and Extension,Wrist Wizard,21,3,Advanced,55.0,77.0,10.0
1,1,Yasmine,Bouzid,8,Forearm Rotation,Wrist Wizard,23,2,Intermediate,70.0,72.0,7.5
2,3,Fatma,Tlili,8,Forearm Rotation,Wrist Wizard,22,1,Beginner,45.0,64.0,5.0
3,20,Omar,Dridi,1,Forward Arm Raise,Bubble Reach,2,2,Intermediate,120.0,70.0,7.5
4,11,Sirine,Ben Salah,5,Mini Squat,Step Quest,14,2,Intermediate,45.0,71.0,7.5
5,17,Skander,Karoui,3,Overhead Reach,Bubble Reach,7,1,Beginner,120.0,66.0,5.0
6,9,Adam,Mejri,6,Step Up,Step Quest,18,3,Advanced,100.0,81.0,10.0
7,19,Mohamed,Slimani,4,Seated Knee Extension,Step Quest,12,3,Advanced,90.0,76.0,10.0
8,11,Sirine,Ben Salah,6,Step Up,Step Quest,17,2,Intermediate,80.0,73.0,7.5
9,20,Omar,Dridi,6,Step Up,Step Quest,18,3,Advanced,100.0,81.0,10.0


In [36]:
print(patient_exercises_analysis.shape)
print(patient_exercises_analysis.columns.tolist())

(43, 26)
['id', 'patient_id', 'exercise_id', 'start_level_id', 'current_level_id', 'assigned_by', 'assigned_at', 'is_active', 'id_patient', 'first_name', 'last_name', 'date_of_birth', 'id_exercise', 'game_id', 'name', 'description', 'id_game', 'name_game', 'body_part', 'description_game', 'id_current_level', 'level_number', 'title', 'target_angle', 'required_score', 'required_progression']


In [37]:
important_columns = [
    "patient_id",
    "exercise_id",
    "game_id",
    "current_level_id",
    "first_name",
    "name",
    "name_game",
    "level_number"
]

print(
    patient_exercises_analysis[important_columns]
    .isnull()
    .sum()
)

patient_id          0
exercise_id         0
game_id             0
current_level_id    0
first_name          0
name                0
name_game           0
level_number        0
dtype: int64


In [38]:
display(
    patient_exercises_analysis[
        [
            "patient_id",
            "first_name",
            "last_name",
            "exercise_id",
            "name",
            "game_id",
            "name_game",
            "body_part",
            "current_level_id",
            "level_number",
            "title"
        ]
    ].head(10)
)

,patient_id,first_name,last_name,exercise_id,name,game_id,name_game,body_part,current_level_id,level_number,title
0,1,Yasmine,Bouzid,7,Wrist Flexion and Extension,3,Wrist Wizard,Wrist,21,3,Advanced
1,1,Yasmine,Bouzid,8,Forearm Rotation,3,Wrist Wizard,Wrist,23,2,Intermediate
2,3,Fatma,Tlili,8,Forearm Rotation,3,Wrist Wizard,Wrist,22,1,Beginner
3,20,Omar,Dridi,1,Forward Arm Raise,1,Bubble Reach,Shoulder,2,2,Intermediate
4,11,Sirine,Ben Salah,5,Mini Squat,2,Step Quest,Knee,14,2,Intermediate
5,17,Skander,Karoui,3,Overhead Reach,1,Bubble Reach,Shoulder,7,1,Beginner
6,9,Adam,Mejri,6,Step Up,2,Step Quest,Knee,18,3,Advanced
7,19,Mohamed,Slimani,4,Seated Knee Extension,2,Step Quest,Knee,12,3,Advanced
8,11,Sirine,Ben Salah,6,Step Up,2,Step Quest,Knee,17,2,Intermediate
9,20,Omar,Dridi,6,Step Up,2,Step Quest,Knee,18,3,Advanced


In [39]:
session_analysis = patient_exercises_analysis.merge(
    sessions,
    left_on="id",
    right_on="patient_exercise_id",
    how="inner",
    suffixes=("_assignment", "_session")
)

In [40]:
print("Nombre de sessions originales :", sessions.shape[0])
print("Nombre de lignes après merge :", session_analysis.shape[0])

Nombre de sessions originales : 500
Nombre de lignes après merge : 500


In [41]:
print(
    "Nombre de sessions uniques :",
    session_analysis["id_session"].nunique()
)

Nombre de sessions uniques : 500


In [42]:
session_analysis = session_analysis.merge(
    results[
        [
            "session_id",
            "repetitions",
            "success_rate",
            "progression"
        ]
    ],
    left_on="id_session",
    right_on="session_id",
    how="left"
)

In [43]:
print("Nombre de lignes :", session_analysis.shape[0])
print(
    "Nombre de sessions uniques :",
    session_analysis["id_session"].nunique()
)

Nombre de lignes : 500
Nombre de sessions uniques : 500


In [44]:
print(
    "Sessions sans résultat :",
    session_analysis["repetitions"].isna().sum()
)

Sessions sans résultat : 0


In [45]:
print(session_analysis.columns.tolist())

['id_assignment', 'patient_id', 'exercise_id', 'start_level_id', 'current_level_id', 'assigned_by', 'assigned_at', 'is_active', 'id_patient', 'first_name', 'last_name', 'date_of_birth', 'id_exercise', 'game_id', 'name', 'description', 'id_game', 'name_game', 'body_part', 'description_game', 'id_current_level', 'level_number', 'title', 'target_angle', 'required_score', 'required_progression', 'id_session', 'patient_exercise_id', 'exercise_level_id', 'session_date', 'duration', 'score', 'session_id', 'repetitions', 'success_rate', 'progression']


In [46]:
display(
    session_analysis[
        [
            "id_session",
            "first_name",
            "last_name",
            "name",
            "name_game",
            "level_number",
            "session_date",
            "duration",
            "score",
            "repetitions",
            "success_rate",
            "progression"
        ]
    ].head(10)
)

,id_session,first_name,last_name,name,name_game,level_number,session_date,duration,score,repetitions,success_rate,progression
0,1,Yasmine,Bouzid,Wrist Flexion and Extension,Wrist Wizard,3,2026-04-30 15:50:31,4,60.6,9,60.2,NaN
1,2,Yasmine,Bouzid,Wrist Flexion and Extension,Wrist Wizard,3,2026-05-04 15:00:15,8,52.5,9,52.7,-13.37
2,4,Yasmine,Bouzid,Wrist Flexion and Extension,Wrist Wizard,3,2026-05-07 15:00:09,5,63.8,10,68.7,21.52
3,6,Yasmine,Bouzid,Wrist Flexion and Extension,Wrist Wizard,3,2026-05-12 15:10:22,4,60.3,6,59.0,-5.49
4,8,Yasmine,Bouzid,Wrist Flexion and Extension,Wrist Wizard,3,2026-05-16 10:45:20,9,62.7,11,70.3,3.98
5,10,Yasmine,Bouzid,Wrist Flexion and Extension,Wrist Wizard,3,2026-05-21 15:40:50,7,61.0,14,60.4,-2.71
6,13,Yasmine,Bouzid,Wrist Flexion and Extension,Wrist Wizard,3,2026-05-25 15:20:30,4,68.5,7,72.1,12.30
7,18,Yasmine,Bouzid,Wrist Flexion and Extension,Wrist Wizard,3,2026-05-30 10:15:13,6,68.9,10,55.4,0.58
8,44,Yasmine,Bouzid,Wrist Flexion and Extension,Wrist Wizard,3,2026-06-17 15:40:41,9,65.4,11,58.3,-5.08
9,60,Yasmine,Bouzid,Wrist Flexion and Extension,Wrist Wizard,3,2026-06-20 10:30:31,10,60.3,16,60.6,-7.80


In [47]:
important_session_columns = [
    "id_session",
    "patient_id",
    "exercise_id",
    "game_id",
    "id_current_level",
    "session_date",
    "score",
    "repetitions",
    "success_rate"
]

print(
    session_analysis[important_session_columns]
    .isnull()
    .sum()
)

id_session          0
patient_id          0
exercise_id         0
game_id             0
id_current_level    0
session_date        0
score               0
repetitions         0
success_rate        0
dtype: int64


In [48]:
print(
    "Scores invalides :",
    (
        (session_analysis["score"] < 0) |
        (session_analysis["score"] > 100)
    ).sum()
)

Scores invalides : 0


In [49]:
print(
    "Success rates invalides :",
    (
        (session_analysis["success_rate"] < 0) |
        (session_analysis["success_rate"] > 100)
    ).sum()
)

Success rates invalides : 0


In [50]:
print(
    "Repetitions invalides :",
    (session_analysis["repetitions"] < 0).sum()
)

Repetitions invalides : 0


In [51]:
print(
    "Nombre de patients :",
    session_analysis["patient_id"].nunique()
)

Nombre de patients : 20


In [52]:
print(
    "Nombre d'exercices :",
    session_analysis["exercise_id"].nunique()
)

Nombre d'exercices : 8


In [53]:
print(
    "Nombre de jeux :",
    session_analysis["game_id"].nunique()
)

Nombre de jeux : 3


In [54]:
print(
    "Nombre de niveaux utilisés :",
    session_analysis["level_number"].nunique()
)

Nombre de niveaux utilisés : 3


In [55]:
print(
    "Progressions manquantes :",
    session_analysis["progression"].isna().sum()
)

Progressions manquantes : 43


In [56]:
print(
    "Progressions disponibles :",
    session_analysis["progression"].notna().sum()
)

Progressions disponibles : 457


In [57]:
print("Dimensions finales :", session_analysis.shape)

print("\nColonnes finales :")
for column in session_analysis.columns:
    print("-", column)

Dimensions finales : (500, 36)

Colonnes finales :
- id_assignment
- patient_id
- exercise_id
- start_level_id
- current_level_id
- assigned_by
- assigned_at
- is_active
- id_patient
- first_name
- last_name
- date_of_birth
- id_exercise
- game_id
- name
- description
- id_game
- name_game
- body_part
- description_game
- id_current_level
- level_number
- title
- target_angle
- required_score
- required_progression
- id_session
- patient_exercise_id
- exercise_level_id
- session_date
- duration
- score
- session_id
- repetitions
- success_rate
- progression


In [58]:
exercise_analysis.to_csv(
    PROCESSED_PATH + "exercise_analysis.csv",
    index=False
)

patient_exercises_analysis.to_csv(
    PROCESSED_PATH + "patient_exercise_analysis.csv",
    index=False
)

session_analysis.to_csv(
    PROCESSED_PATH + "session_analysis.csv",
    index=False
)

In [ ]:
print("Fichiers analytiques créés :")

for file in [
    "exercise_analysis.csv",
    "patient_exercise_analysis.csv",
    "session_analysis.csv"
]:
    path = PROCESSED_PATH + file
    
    print(
        f"{file:35} →",
        "OK" if os.path.exists(path) else "ERREUR"
    )

Fichiers analytiques créés :
exercise_analysis.csv               → OK
patient_exercise_analysis.csv       → OK
session_analysis.csv                → OK


: 